# Validation of model_predict_qubit_TransmonCross_cap_matrix wtih Ansys Q3D

In [1]:
from squadds import SQuADDS_DB
import pandas as pd
from squadds import Analyzer
import matplotlib.pyplot as plt
from squadds import AnsysSimulator
import numpy as np

We don't actually care about using SQuADDS to get a device at the moment, we just use the code block below to get a "template" of the SQUaDDS style dataframe.

In [2]:
''' grab SQuADDS entry as a template for ML predicted designs ''' 
db = SQuADDS_DB()
db.select_system("qubit")
db.select_qubit("TransmonCross")
df = db.create_system_df()

analyzer = Analyzer(db)

# we are not actually looking for these Hamiltonian parameters... 
target_params={"qubit_frequency_GHz": 4, "anharmonicity_MHz": -200}

pred_df = analyzer.find_closest(target_params=target_params,
                                       num_top=1,
                                       metric="Euclidean",
                                       display=True)

''' read in ML results ''' 
ML_results = pd.read_csv("predictions_and_errors_unscaled_one_hot.csv") # real in ML test results

## Sweep through ML design results, simulate each design with Anysys Q3D, and save sim results
There are only three multi-valued parameters in the SQuADDS qubit-TransmonCross-cap_matrix dataset:
* connection_pads.readout.claw_length
* connection_pads.readout.ground_spacing
* cross_length

The model only predicts these three design parameters. We take a SQuADDS database entry and substitute in the predicted parameters from the model.

In [3]:
# dictionary to save results in 
results = pd.DataFrame({"Sample":[],
                        "ref_design":[],
                        "pred_design":[],
                        "ref_H_params":[],
                        "ref_cap_matrix":[],
                        "pred_H_params":[],
                        "pred_cap_matrix":[]})

um = 10**6 ## ML model is trained in SI units (m), convert back to µm  
samples_to_test = np.arange(0,50)

In [4]:
for sample in samples_to_test:

    ''' current testing sample '''
    this_device = ML_results[ML_results.sample_idx == sample]

    ''' get ML predicted design parameters '''
    # reference/truth device parameters
    ref_claw_length = str(this_device.ref_unscaled.iloc[0] * um)+'um' # grab device params, convert back to microns, and add unit labels
    ref_ground_spacing = str(this_device.ref_unscaled.iloc[1] * um)+'um'
    ref_cross_length = str(this_device.ref_unscaled.iloc[2] * um)+'um'
    
    # predicted device parameters
    pred_claw_length = str(this_device.pred_unscaled.iloc[0] * um)+'um'
    pred_ground_spacing = str(this_device.pred_unscaled.iloc[1] * um)+'um'
    pred_cross_length = str(this_device.pred_unscaled.iloc[2] * um)+'um'

    ''' create our predicted design option for Qiskit Metal '''
    pred_df.design_options.iloc[0]["connection_pads"]["readout"]["claw_length"] = pred_claw_length
    pred_df.design_options.iloc[0]["connection_pads"]["readout"]["ground_spacing"] = pred_ground_spacing
    pred_df.design_options.iloc[0]["cross_length"] = pred_cross_length
    pred_device = pred_df.iloc[0]

    ''' simulate predicted design '''
    pred_ansys_simulator = AnsysSimulator(analyzer, pred_device)
    pred_ansys_results = pred_ansys_simulator.simulate(pred_device)
    pred_capacitance_matrix = pred_ansys_results["sim_results"]
    
    print("Predicted Hamiltonian parameters:")
    pred_Hamiltonian_params = pred_ansys_simulator.get_xmon_info(pred_ansys_results) # get Hamiltonian parameters

    ''' save results '''
    ref_design = {"claw_length":ref_claw_length,"ground_spacing":ref_ground_spacing,"cross_length":ref_cross_length}
    pred_design = {"claw_length":pred_claw_length,"ground_spacing":pred_ground_spacing,"cross_length":pred_cross_length}

    # reference/truth capacitance parameters
    ref_capacitance_matrix = {"cross_to_ground":this_device.cross_to_ground.iloc[0],
                              "claw_to_ground":this_device.claw_to_ground.iloc[0],
                              "cross_to_claw":this_device.cross_to_claw.iloc[0],
                              "cross_to_cross":this_device.cross_to_cross.iloc[0],
                              "claw_to_claw":this_device.claw_to_claw.iloc[0],
                              "ground_to_ground":this_device.ground_to_ground.iloc[0],
                              "units":"fF"}
    
    # we don't want to use compute to simulate the pre-simulated design from SQuADDS, instead we make faux results to back out the Hamiltonian
    ref_ansys_simulator = AnsysSimulator(analyzer,None)
    ref_ansys_results = {"design": pred_ansys_results["design"], # only extracts Lj, doesn't change from pred & ref
                         "sim_options": pred_ansys_results["sim_options"], # only extracts Lj, doesn't change from pred & ref
                         "sim_results": ref_capacitance_matrix}
    
    print("Reference Hamiltonian parameters:")
    ref_Hamiltonian_params = ref_ansys_simulator.get_xmon_info(ref_ansys_results)
    
    results.loc[len(results)] = [sample,
                                 ref_design,
                                 pred_design,
                                 ref_Hamiltonian_params,
                                 ref_capacitance_matrix,
                                 pred_Hamiltonian_params,
                                 pred_capacitance_matrix]

INFO 12:50PM [connect_project]: Connecting to Ansys Desktop API...
INFO 12:50PM [load_ansys_project]: 	Opened Ansys App
INFO 12:50PM [load_ansys_project]: 	Opened Ansys Desktop v2023.2.0
INFO 12:50PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/firas/Documents/Ansoft/
	Project:   Project7
INFO 12:50PM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d3 [Solution type: Q3D]
INFO 12:50PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 12:50PM [connect]: 	Connected to project "Project7" and design "LOMv2.0_q3d3" 😀 



selected system: qubit
the parameters ['run'] are unsupported, so they have been ignored


INFO 12:50PM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d4 [Solution type: Q3D]
WARNING 12:50PM [connect_setup]: 	No design setup detected.
WARNING 12:50PM [connect_setup]: 	Creating Q3D default setup.
INFO 12:50PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 12:50PM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 12:50PM [analyze]: Analyzing setup sweep_setup
INFO 12:56PM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmphloep78k.txt, C, , sweep_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 12:56PM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmpbzhy64nl.txt, C, , sweep_setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 12:56PM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmpzctudgfx.txt, C, , sweep_setup:AdaptiveP

Predicted Hamiltonian parameters:
qubit anharmonicity = -150 MHz 
qubit frequency = 4.109 GHz
selected system: qubit
Reference Hamiltonian parameters:
qubit anharmonicity = -151 MHz 
qubit frequency = 4.115 GHz
selected system: qubit
the parameters ['run'] are unsupported, so they have been ignored


INFO 12:56PM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d5 [Solution type: Q3D]
WARNING 12:56PM [connect_setup]: 	No design setup detected.
WARNING 12:56PM [connect_setup]: 	Creating Q3D default setup.
INFO 12:56PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 12:56PM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 12:56PM [analyze]: Analyzing setup sweep_setup
INFO 01:03PM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmpezaulzjc.txt, C, , sweep_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 01:03PM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmpia603gfg.txt, C, , sweep_setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 01:03PM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmpzhqlksua.txt, C, , sweep_setup:AdaptiveP

...

Predicted Hamiltonian parameters:
qubit anharmonicity = -112 MHz 
qubit frequency = 3.58 GHz
selected system: qubit
Reference Hamiltonian parameters:
qubit anharmonicity = -116 MHz 
qubit frequency = 3.649 GHz
selected system: qubit
the parameters ['run'] are unsupported, so they have been ignored


INFO 06:21PM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d51 [Solution type: Q3D]
INFO 06:21PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 06:21PM [connect]: 	Connected to project "Project7" and design "LOMv2.0_q3d51" 😀 

INFO 06:21PM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d52 [Solution type: Q3D]
WARNING 06:21PM [connect_setup]: 	No design setup detected.
WARNING 06:21PM [connect_setup]: 	Creating Q3D default setup.
INFO 06:21PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 06:21PM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 06:21PM [analyze]: Analyzing setup sweep_setup
INFO 06:30PM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmphx97_0im.txt, C, , sweep_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 06:30PM [get_matrix]: Exporting matrix data to (C:\Users\fi

Predicted Hamiltonian parameters:
qubit anharmonicity = -171 MHz 
qubit frequency = 4.363 GHz
selected system: qubit
Reference Hamiltonian parameters:
qubit anharmonicity = -172 MHz 
qubit frequency = 4.373 GHz
selected system: qubit
the parameters ['run'] are unsupported, so they have been ignored


INFO 06:30PM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d52 [Solution type: Q3D]
INFO 06:30PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 06:30PM [connect]: 	Connected to project "Project7" and design "LOMv2.0_q3d52" 😀 

INFO 06:30PM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d53 [Solution type: Q3D]
WARNING 06:30PM [connect_setup]: 	No design setup detected.
WARNING 06:30PM [connect_setup]: 	Creating Q3D default setup.
INFO 06:30PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 06:30PM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 06:30PM [analyze]: Analyzing setup sweep_setup
INFO 06:39PM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmp7ujyx9s5.txt, C, , sweep_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 06:39PM [get_matrix]: Exporting matrix data to (C:\Users\fi

Predicted Hamiltonian parameters:
qubit anharmonicity = -128 MHz 
qubit frequency = 3.816 GHz
selected system: qubit
Reference Hamiltonian parameters:
qubit anharmonicity = -134 MHz 
qubit frequency = 3.892 GHz


## Save the data

In [5]:
results.to_csv("transmonCapMat_simulation_results.csv",index=False) # save data for later analysis with validation_01_data_analysis.ipynb